# ⭐ Initial Set up

**IMPORTANT**
To get fastly to the explainability analysis, extract the files contained in the .zip into the ml_model directory. Then, run the sections marked with ⭐ ( Initial set up, Checkpoint - Load Data and Model, Model Entry Point, Checkpoint - Load data transformed). After that, you can continue with the local explanations section

In [1]:
import os
import import_ipynb
from mimic_utils_text import InHospitalMortalityReader, read_chunk
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

importing Jupyter notebook from mimic_utils_text.ipynb


c:\Users\DAIMA Researcher\anaconda3\Lib\site-packages\nbformat\__init__.py:93: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [2]:
padding_num_features = 35
np.random.seed(42)

## Set logger

In [3]:
import logging

def setup_logging():
    LOGGER = logging.getLogger("LSTM")

    LOGGER.setLevel(logging.INFO)
    logging_format = logging.Formatter(
        "[%(asctime)s - %(filename)s:%(lineno)s - %(funcName)s() - %(levelname)s %(message)s"
    )
    ch = logging.StreamHandler()
    ch.setFormatter(logging_format)
    LOGGER.addHandler(ch)
    return LOGGER

LOGGER = setup_logging()

## Set basic functions & classes

In [4]:
# pad missing values in the nested lists with 0s
def pad_missing_value_with_zero(data):
    a1 = np.zeros((len(data), max([len(k) for k in data]), padding_num_features))  # 35
    for ctr, k in enumerate(data):
        # print(ctr, len(k), k)
        # Convert string representations of Boolean values to numerical values
        k = np.array([[1.0 if val == "True" else 0.0 if val == "False" else float(val) for val in row] for row in k])

        a1[ctr, : len(k), :] = k
    return a1

In [5]:
# pytroch class for reading data into batches
class MIMICDataset(Dataset):
    """
    Loads time series data into memory from a text file,
    split by newlines.
    """

    def __init__(self, reader, target_repl=False, batch_labels=False):
        self.data = []
        self.y = []
        N = reader.get_number_of_examples()
        print(f"Number of examples:{N}")
        # read data form cvs files
        ret = read_chunk(reader, N)
        # read into memory structured data X and labels y
        # print(ret)

        data = ret["X"]
        ts = ret["t"]
        labels = ret["y"]
        names = ret["name"]
        self.features = ret["header"]

        # print(data)

        #Filtering data
        data, labels = zip(*[(d, l) for d, l in zip(data, labels) if len(d) == 48])
        data, labels = list(data), list(labels)  # Convertir de nuevo a listas

        

        # pad missing values in the list of arrays with 0s
        data = pad_missing_value_with_zero(data)

        self.data = np.array(data, dtype=np.float32)
        self.T = self.data.shape[1]

        if batch_labels:
            self.y = np.array([[l] for l in labels], dtype=np.float32)
        else:
            self.y = np.array(labels, dtype=np.float32)
        if target_repl:
            self.y = self._extend_labels(self.y)

    def _extend_labels(self, labels):
        # (B,)
        labels = labels.repeat(self.T, axis=1)  # (B, T)
        return labels

    def __len__(self):
        # overide len to get number of instances
        return len(self.data)

    def __getitem__(self, idx):
        # get features (physiological variables x) and label for a given instance index
        return self.data[idx], self.y[idx]

In [6]:
# model
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        tag_size,
        hidden_size,
        feat_size,
        emb_size,
        bidirectional=False,
        dropout=0.2,
        aggregation_type="last_state",
    ):
        """
        constructor, here we define the hidden layers for our architecture
        """
        super().__init__()

        # define if the rnn will be bidirectional
        self.bidirectional = bidirectional

        # define the aggregation type of the features for the classifier for example, you can take the mean
        self.aggregation_type = aggregation_type
        self.encoder = nn.Linear(feat_size, emb_size, bias=True)
        
        # Create a (bidirectional) LSTM to encode sequence
        self.lstm = nn.LSTM(emb_size, hidden_size, batch_first=True, bidirectional=bidirectional)

        # The output of the LSTM doubles if we use a bidirectional encoder.
        encoding_size = hidden_size * 2 if bidirectional else hidden_size
        self.combination_layer = nn.Linear(encoding_size, encoding_size)

        # Create affine layer to project to the classes
        self.projection = nn.Linear(encoding_size, tag_size)
        
        # dropout layer for regularizetion of a sequence
        self.dropout_layer = nn.Dropout(p=dropout)
        self.relu = nn.ReLU()

    def forward(self, x, seq_mask=None, seq_len=None):
        # input size
        # [B, T, feat_size] batch, time and features
        # return unormalized probabilities (logits)
        # the loss will compute the sigmoid and negative log-likelihood
        # output size
        # [B, num_class] batch, and 1 class

        # [B, T, F] batch, time, features
        h1 = self.encoder(x)
        h1 = self.relu(h1)
        # [B, T, H] batch, time, hidden or hidden * 2
        outputs, (final, _) = self.lstm(h1)

        if self.aggregation_type == "mean":
            # mean over hidden states of LSTM
            outputs = self.dropout_layer(outputs)
            h = self.relu(self.combination_layer(outputs))
            # [B, H] batch, hidden
            h = h.mean(dim=1)  # mean over time dimension
        elif self.aggregation_type == "last_state":
            # last hidden state of the lstm or concat of bidirectional forward and backward states
            if self.bidirectional:
                h_T_fwd = final[0]  # lstm 1, last hidden state of forward lstm
                h_T_bwd = final[1]  # lstm 2. last hidden state of backward lstm
                # [B, H*2]
                h = torch.cat(
                    [h_T_fwd, h_T_bwd], dim=-1
                )  # concatenate the forward with the backward in the last dimension (feat)
            else:
                h = final[-1]
            h = self.relu(self.combination_layer(h))
            h = self.dropout_layer(h)
        # [B, H] # summary for each patient
        # [B, 1]
        logits = self.projection(h)

        return logits

In [7]:
def load_model():
    # Define the classification model.
    model = LSTMClassifier(
        tag_size=1,  # binary
        feat_size=feat_size,
        hidden_size=args["dim"],
        emb_size=args["emb_size"],
        bidirectional=args["bidirectional"],
        dropout=args["dropout"],
        aggregation_type=args["aggregation_type"],
    )

    # load trained model from file
    model.load_state_dict(torch.load(args["best_model"]))
    LOGGER.info(model)

    device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    model = model.to(device)
    return model

In [8]:
data_dir = "data/AKI/fts_extract_race_groups"
model_path = 'data/models/2024-12-13/fts_extract_race_groups/_dropout_0.2,batch_size_64,lr_0.pth'

args = {
    "best_model": model_path,
    "dim": 35,
    "dropout": 0.2,
    "batch_size": 16,
    "emb_size": 35,
    "aggregation_type": "mean",
    "bidirectional": False,
    "data": data_dir,  # path to data
    "notes": data_dir,  # the code ignores the text
    "timestep": 1.0,
    "imputation": "previous",
    "normalizer_state": None,
}

# Load data

In [60]:
# Load training data
train_reader = InHospitalMortalityReader(
    dataset_dir=os.path.join(args['data'], "train"),
    notes_dir=args['notes'],
    listfile=os.path.join(args['notes'], "train_listfile.csv"),
    period_length=48.0,
)

train_dataset = MIMICDataset(train_reader, batch_labels=True)
train_dl = DataLoader(train_dataset, batch_size=100, shuffle=False)

InHospitalMortalityReader init completed
Number of examples:51996
Reading chunk of size 51996
Number of records with more than 48 hours: 434


In [ ]:
# # Save train_dataset
# torch.save(train_dataset, 'train_dataset.pth')
# torch.save(train_dl, 'train_dataloader.pth')

In [ ]:
# # load
# train_dataset = torch.load('train_dataset.pth')
# train_dl = torch.load('train_dataloader.pth')

In [ ]:
test_reader = InHospitalMortalityReader(
    dataset_dir=os.path.join(args['data'], "test"),
    notes_dir=args['notes'],
    listfile=os.path.join(args['notes'], "test_listfile.csv"),
    period_length=48.0,
)

test_dataset = MIMICDataset(test_reader, batch_labels=True)
test_dl = DataLoader(test_dataset, batch_size=100, shuffle=False)
# [B, M, feat_size]

InHospitalMortalityReader init completed
Number of examples:6368
Reading chunk of size 6368
Number of records with more than 48 hours: 56


In [ ]:
# torch.save(test_dataset, 'test_dataset.pth')
# torch.save(test_dl, 'test_dl.pth')

In [30]:
# load
test_dataset = torch.load('test_dataset.pth')
test_dl = torch.load('test_dl.pth')

In [16]:
feat_size = test_dataset.data.shape[-1]
test_dataset.data.shape, len(test_dataset.y)

NameError: name 'test_dataset' is not defined

# Load Model

In [ ]:
model = load_model()
model

# ⭐ Checkpoint - Load Data and Model

In [9]:
# load
train_dataset = torch.load('train_dataset.pth')
train_dl = torch.load('train_dataloader.pth')
test_dataset = torch.load('test_dataset.pth')
test_dl = torch.load('test_dl.pth')
feat_size = test_dataset.data.shape[-1]

model = load_model()

[2025-02-14 01:34:57,487 - 3221639268.py:15 - load_model() - INFO LSTMClassifier(
  (encoder): Linear(in_features=35, out_features=35, bias=True)
  (lstm): LSTM(35, 35, batch_first=True)
  (combination_layer): Linear(in_features=35, out_features=35, bias=True)
  (projection): Linear(in_features=35, out_features=1, bias=True)
  (dropout_layer): Dropout(p=0.2, inplace=False)
  (relu): ReLU()
)


# TimeSHAP

In [15]:
!pip install timeshap

In [11]:
from timeshap import __version__
__version__

'1.0.4'

## ⭐ Model Entry Point

In [11]:
from timeshap.wrappers import TorchModelWrapper
model_wrapped = TorchModelWrapper(model)
f_hs = lambda x, y=None: model_wrapped.predict_last_hs(x, y)

## Transform data to the format required

In [65]:
test_dataset.data.shape, train_dataset.data.shape

((5952, 48, 35), (48528, 48, 35))

In [66]:
df_traindata = pd.DataFrame(train_dataset.data.reshape(-1, train_dataset.data.shape[-1]), columns=train_dataset.features)
df_traindata['instance'] = np.repeat(np.arange(len(train_dataset)), train_dataset.data.shape[1])
df_traindata['label'] = np.repeat(train_dataset.y, train_dataset.data.shape[1])
df_traindata

,Hours,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,hematocrit_avg,...,black_african_group,asian_group,hispanic_group,unknown_group,native_group,other_group,ELECTIVE,URGENT,instance,label
0,0.466667,0.631579,0.423077,0.225490,0.470588,0.095238,0.746479,0.254844,0.731707,0.462094,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0
1,1.300000,0.631579,0.423077,0.147059,0.441176,0.095238,0.225352,0.329359,0.536585,0.418773,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0
2,2.300000,0.631579,0.423077,0.147059,0.441176,0.095238,0.732394,0.648286,0.682927,0.418773,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0
3,3.300000,0.631579,0.423077,0.147059,0.441176,0.095238,0.732394,0.648286,0.902439,0.418773,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0
4,4.300000,0.631579,0.423077,0.147059,0.441176,0.095238,0.605634,0.648286,0.548781,0.418773,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2329339,43.258057,0.578947,0.461538,1.000000,0.705882,0.309524,0.239437,0.275708,0.207317,0.299639,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48527,0.0
2329340,44.258057,0.578947,0.461538,1.000000,0.705882,0.309524,0.239437,0.275708,0.207317,0.299639,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48527,0.0
2329341,45.258057,0.578947,0.461538,1.000000,0.705882,0.309524,0.239437,0.275708,0.207317,0.299639,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48527,0.0
2329342,46.258057,0.578947,0.461538,1.000000,0.705882,0.309524,0.239437,0.275708,0.207317,0.299639,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48527,0.0


In [39]:
df_testdata = pd.DataFrame(test_dataset.data.reshape(-1, test_dataset.data.shape[-1]), columns=test_dataset.features)
df_testdata['instance'] = np.repeat(np.arange(len(test_dataset)), test_dataset.data.shape[1])
df_testdata['label'] = np.repeat(test_dataset.y, test_dataset.data.shape[1])
df_testdata

,Hours,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,hematocrit_avg,...,black_african_group,asian_group,hispanic_group,unknown_group,native_group,other_group,ELECTIVE,URGENT,instance,label
0,0.466667,0.210526,0.346154,0.460784,1.000000,0.107143,0.661972,0.120715,0.304878,0.462094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
1,1.133333,0.210526,0.346154,0.460784,1.000000,0.107143,0.563380,0.120715,0.329268,0.462094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
2,2.133333,0.210526,0.346154,0.460784,1.000000,0.107143,0.366197,0.120715,0.463415,0.462094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
3,3.133333,0.210526,0.346154,0.460784,1.000000,0.107143,0.450704,0.120715,0.280488,0.462094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
4,4.133333,0.210526,0.346154,0.460784,1.000000,0.107143,0.239437,0.120715,0.304878,0.462094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285691,43.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.323944,0.643815,0.719512,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0
285692,44.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.323944,0.475410,0.719512,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0
285693,45.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.619718,0.643815,0.719512,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0
285694,46.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.591549,0.643815,0.682927,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0


In [ ]:
# Save
# df_traindata.to_csv('df_traindata.csv', index=False)
# df_testdata.to_csv('df_testdata.csv', index=False)

## ⭐ Checkpoint - Load data transformed

In [12]:
# Load
df_traindata = pd.read_csv('df_traindata.csv')
df_testdata = pd.read_csv('df_testdata.csv')

In [13]:
# df_testdata.iloc[90:100]
df_testdata.tail()

,Hours,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,hematocrit_avg,...,black_african_group,asian_group,hispanic_group,unknown_group,native_group,other_group,ELECTIVE,URGENT,instance,label
285691,43.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.323944,0.643815,0.719512,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0
285692,44.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.323944,0.475410,0.719512,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0
285693,45.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.619718,0.643815,0.719512,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0
285694,46.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.591549,0.643815,0.682927,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0
285695,47.371113,0.210526,0.384615,0.107843,0.529412,0.071429,0.352113,0.612519,0.756098,0.407942,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,5951,1.0


In [14]:
model_features = test_dataset.features
raw_model_features = test_dataset.features
print(len(model_features))
model_features

35


['Hours',
 'aniongap_avg',
 'bicarbonate_avg',
 'bun_avg',
 'chloride_avg',
 'creat',
 'diasbp_mean',
 'glucose_avg',
 'heartrate_mean',
 'hematocrit_avg',
 'hemoglobin_avg',
 'potassium_avg',
 'resprate_mean',
 'sodium_avg',
 'spo2_mean',
 'sysbp_mean',
 'uo_rt_12hr',
 'uo_rt_24hr',
 'uo_rt_6hr',
 'wbc_avg',
 'sedative',
 'vasopressor',
 'vent',
 'anchor_age',
 'F',
 'M',
 'white_group',
 'black_african_group',
 'asian_group',
 'hispanic_group',
 'unknown_group',
 'native_group',
 'other_group',
 'ELECTIVE',
 'URGENT']

In [15]:
categorical_features = ['sedative','vasopressor','vent','F','M','white_group','ELECTIVE','URGENT','black_african_group','asian_group','hispanic_group','unknown_group','native_group','other_group']
numerical_features = [model_features[i] for i in range(len(model_features)) if model_features[i] not in categorical_features]

print(len(categorical_features))
print(len(numerical_features))

14
21


In [16]:
plot_features = {feat: feat for feat in test_dataset.features}
plot_features

{'Hours': 'Hours',
 'aniongap_avg': 'aniongap_avg',
 'bicarbonate_avg': 'bicarbonate_avg',
 'bun_avg': 'bun_avg',
 'chloride_avg': 'chloride_avg',
 'creat': 'creat',
 'diasbp_mean': 'diasbp_mean',
 'glucose_avg': 'glucose_avg',
 'heartrate_mean': 'heartrate_mean',
 'hematocrit_avg': 'hematocrit_avg',
 'hemoglobin_avg': 'hemoglobin_avg',
 'potassium_avg': 'potassium_avg',
 'resprate_mean': 'resprate_mean',
 'sodium_avg': 'sodium_avg',
 'spo2_mean': 'spo2_mean',
 'sysbp_mean': 'sysbp_mean',
 'uo_rt_12hr': 'uo_rt_12hr',
 'uo_rt_24hr': 'uo_rt_24hr',
 'uo_rt_6hr': 'uo_rt_6hr',
 'wbc_avg': 'wbc_avg',
 'sedative': 'sedative',
 'vasopressor': 'vasopressor',
 'vent': 'vent',
 'anchor_age': 'anchor_age',
 'F': 'F',
 'M': 'M',
 'white_group': 'white_group',
 'black_african_group': 'black_african_group',
 'asian_group': 'asian_group',
 'hispanic_group': 'hispanic_group',
 'unknown_group': 'unknown_group',
 'native_group': 'native_group',
 'other_group': 'other_group',
 'ELECTIVE': 'ELECTIVE',
 'UR

# Local Explanations

In [21]:
time_feat = 'Hours'
label_feat = 'label'
sequence_id_feat = 'instance'

## Baseline event

In [22]:
from typing import List, Union, Callable, Dict, Tuple
from scipy import stats

## Original function copied and modified
def calc_avg_event(data: Union[pd.DataFrame, np.ndarray],
                   numerical_feats: List[Union[str, int]],
                   categorical_feats: List[Union[str, int]],
                   model_features: List[str] = None,
                   ) -> pd.DataFrame:
    """
    Calculates the average event of a dataset. This event is repeated N times
    to form the background sequence to be used in TimeSHAP.

    Calculates the median of numerical features, and the mode for categorical
    features of a pandas DataFrame

    Parameters
    ----------
    data: pd.DataFrame
        Dataset to use for baseline calculation

    numerical_feats: List[Union[str, int]]
        List of numerical features or corresponding indexes to calculate median of

    categorical_feats: List[Union[str, int]]
        List of numerical features or corresponding indexes to calculate mode of

    model_features: List[str]
        Model features to infer the indexes of schema. Needed when using strings to identify features

    Returns
    -------
    pd.DataFrame
        DataFrame with the median/mode of the features
    """
    if len(numerical_feats) > 0 and isinstance(numerical_feats[0], str) or  len(categorical_feats) > 0 and isinstance(categorical_feats[0], str):
        # given features are not indexes
        if isinstance(data, pd.DataFrame):
            model_features = list(data.columns)
        else:
            assert model_features is not None and len(model_features), "When using feature names to identify them, specify the model features. Alternatively you can pass the indexes of the features directly"
        numerical_indexes = [model_features.index(x) for x in numerical_feats]
        categorical_indexes = [model_features.index(x) for x in categorical_feats]
        ordered_feats = numerical_feats + categorical_feats
    else:
        numerical_indexes = numerical_feats
        categorical_indexes = categorical_feats
        ordered_feats = numerical_indexes + categorical_indexes

    if len(data.shape) == 3:
        data = np.squeeze(data, axis=1)
    elif len(data.shape) == 2:
        data = data.values
    else:
        raise ValueError

    numerical = np.median(data[:,  numerical_indexes], axis=0)
    if len(categorical_indexes) > 0:
        # categorical = stats.mode(data[:, categorical_indexes], axis=0)[0][0, :]
        categorical = stats.mode(df_testdata.values[:, categorical_indexes], axis=0)[0]
        numerical = np.concatenate((numerical,categorical), axis=0)

    return pd.DataFrame([numerical], columns=ordered_feats)

In [23]:
# from timeshap.utils import calc_avg_event
average_event = calc_avg_event(df_traindata, numerical_feats=numerical_features, categorical_feats=categorical_features)

In [24]:
average_event

,Hours,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,hematocrit_avg,...,M,white_group,ELECTIVE,URGENT,black_african_group,asian_group,hispanic_group,unknown_group,native_group,other_group
0,23.999861,0.368421,0.5,0.147059,0.558824,0.083333,0.394366,0.198212,0.414634,0.444043,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Baseline sequence

In [22]:
def calc_avg_sequence_v2(data: Union[pd.DataFrame, np.ndarray],
                      numerical_feats: List[Union[str, int]],
                      categorical_feats: List[Union[str, int]],
                      model_features: List[str] = None,
                      entity_col: str = None,
                      ) -> np.ndarray:
    """
    Calculates the average sequence of a dataset. Requires all sequences of the
    dataset to be the same size and ordered by time.

    Calculates the median of numerical features, and the mode for categorical
    features of a pandas DataFrame

    Parameters
    ----------
    data: Union[pd.DataFrame, np.ndarray]
        Dataset to use for baseline calculation

    numerical_feats: List[Union[str, int]]
        List of numerical features or corresponding indexes to calculate median of

    categorical_feats: List[Union[str, int]]
        List of numerical features or corresponding indexes to calculate mode of

    model_features: List[str]
        Model features to infer the indexes of schema. Needed when using strings to identify features

    entity_col: str
        Entity column to identify sequences

    Returns
    -------
    np.ndarray
        Average sequence to use in TimeSHAP
    """
    if isinstance(data, pd.DataFrame):
        assert entity_col is not None, "To calculate average sequence from DataFrame, entity_col is required"
        sequences = data.groupby(entity_col)
        seq_lens = sequences.size().values
        assert np.array([x == seq_lens[0] for x in seq_lens]).all(), "All sequences must be the same length"
        data = np.array([x[1].values for x in sequences])
    elif isinstance(data, np.ndarray) and len(data.shape) == 3:
        pass
    else:
        raise ValueError("Unrecognized data format")
    if len(numerical_feats) > 0 and isinstance(numerical_feats[0], str) or  len(categorical_feats) > 0 and isinstance(categorical_feats[0], str):
        # given features are not indexes
        assert model_features is not None and len(model_features), "When using feature names to identify them, specify the model features. Alternatively you can pass the indexes of the features directly"
        numerical_indexes = [model_features.index(x) for x in numerical_feats]
        categorical_indexes = [model_features.index(x) for x in categorical_feats]
    else:
        numerical_indexes = numerical_feats
        categorical_indexes = categorical_feats

    numerical = np.median(data[:, :,  numerical_indexes], axis=0)
    print(numerical.shape)
    if len(categorical_indexes) > 0:
        # categorical = stats.mode(data[:, :,  categorical_indexes], axis=0)[0][0, :, :]
        categorical = stats.mode(data[:, categorical_indexes], axis=0)[0]
        print(categorical.shape)
        numerical = np.concatenate((numerical, categorical), axis=1)
    return numerical.astype(float)

In [ ]:
# from timeshap.utils import calc_avg_sequence
average_sequence = calc_avg_sequence_v2(df_testdata,
                                     numerical_feats=numerical_features,
                                     categorical_feats=categorical_features,
                                     model_features=model_features,
                                     entity_col=sequence_id_feat)

## Select instance to explain

In [35]:
from timeshap.explainer import local_pruning, local_event, local_feat, local_cell_level
from timeshap.plot import plot_temp_coalition_pruning, plot_event_heatmap, plot_feat_barplot, plot_cell_level

### Positive instance

In [20]:
positive_instance_id = 10
positive_instance = df_testdata[df_testdata['instance'] == positive_instance_id]
positive_instance.head()

,Hours,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,hematocrit_avg,...,black_african_group,asian_group,hispanic_group,unknown_group,native_group,other_group,ELECTIVE,URGENT,instance,label
480,0.572778,0.473684,0.346154,0.156863,0.500000,0.130952,0.169014,0.362146,0.378049,0.238267,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,10,1.0
481,1.572778,0.473684,0.346154,0.156863,0.529412,0.130952,0.169014,0.257824,0.378049,0.238267,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,10,1.0
482,2.572778,0.157895,0.461538,0.156863,0.588235,0.071429,0.165493,0.186289,0.000000,0.270758,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,10,1.0
483,3.572778,0.473684,0.346154,0.156863,0.529412,0.130952,0.164990,0.257824,0.000000,0.346570,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,10,1.0
484,4.572778,0.473684,0.346154,0.156863,0.529412,0.130952,0.126761,0.257824,0.060976,0.328520,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,10,1.0


In [29]:
# select model features only
positive_instance_data = positive_instance[model_features]
# convert the instance to numpy so TimeSHAP receives it
positive_instance_data = np.expand_dims(positive_instance_data.to_numpy().copy(), axis=0)

In [96]:
pruning_dict = {'tol': 9}
coal_plot_data, coal_prun_idx = local_pruning(f_hs, positive_instance_data, pruning_dict, average_event, positive_instance_id, sequence_id_feat, False)
# coal_prun_idx is in negative terms
pruning_idx = positive_instance_data.shape[1] + coal_prun_idx

In [97]:
positive_instance_data.shape, coal_prun_idx, pruning_idx

((1, 48, 35), -48, 0)

In [98]:
coal_plot_data.sort_values(by='Shapley Value', ascending=False)

,Coalition,t (event index),Shapley Value
7,Sum of contribution of events ≤ t,-3,21.564392
5,Sum of contribution of events ≤ t,-2,21.502146
3,Sum of contribution of events ≤ t,-1,21.501484
96,Sum of contribution of events > t,-48,21.485952
1,Sum of contribution of events ≤ t,0,21.485952
...,...,...,...
0,Sum of contribution of events > t,0,0.000000
97,Sum of contribution of events ≤ t,-48,0.000000
2,Sum of contribution of events > t,-1,-0.015532
4,Sum of contribution of events > t,-2,-0.016195


In [99]:
number_of_events = positive_instance_data.shape[1]
pruning_plot = plot_temp_coalition_pruning(coal_plot_data, coal_prun_idx, plot_limit=number_of_events)
pruning_plot

the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.


alt.LayerChart(...)

In this plot we can see the importance of the grouped events to the instance as we go backwards on the sequence. The lower the importance, the less relevant these events are which means they can be pruned when a certain threshold is reached.

(Before) 0.25 can be the threshold
(After) 12.5

### Negative instance

In [42]:
negative_instance_id = 13
negative_instance = df_testdata[df_testdata['instance'] == negative_instance_id]
negative_instance.head()

,Hours,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,hematocrit_avg,...,black_african_group,asian_group,hispanic_group,unknown_group,native_group,other_group,ELECTIVE,URGENT,instance,label
624,0.466667,0.0,0.423077,0.029412,0.617647,0.035714,0.295775,0.123696,0.463415,0.743682,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13,0.0
625,1.261667,0.0,0.423077,0.029412,0.617647,0.035714,0.295775,0.160954,0.463415,0.140794,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13,0.0
626,2.261667,0.0,0.423077,0.029412,0.617647,0.035714,0.295775,0.160954,0.463415,0.140794,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13,0.0
627,3.261667,0.0,0.423077,0.029412,0.617647,0.035714,0.295775,0.160954,0.463415,0.184116,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13,0.0
628,4.261667,0.0,0.423077,0.029412,0.764706,0.035714,0.295775,0.162444,0.463415,0.140794,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,13,0.0


In [44]:
# Now, let's plot the negative instance
negative_instance_data = negative_instance[model_features]
negative_instance_data = np.expand_dims(negative_instance_data.to_numpy().copy(), axis=0)

pruning_negative_dict = {'tol': 0.2}
coal_plot_negative_data, coal_prun_negative_idx = local_pruning(f_hs, negative_instance_data, pruning_negative_dict, average_event, negative_instance_id, sequence_id_feat, False)
# coal_prun_idx is in negative terms
pruning_negative_idx = negative_instance_data.shape[1] + coal_prun_negative_idx

In [45]:
coal_plot_negative_data.sort_values(by='Shapley Value', ascending=False)

,Coalition,t (event index),Shapley Value
0,Sum of contribution of events > t,0,0.000000
97,Sum of contribution of events ≤ t,-48,0.000000
2,Sum of contribution of events > t,-1,-0.018831
4,Sum of contribution of events > t,-2,-0.036795
6,Sum of contribution of events > t,-3,-0.061954
...,...,...,...
7,Sum of contribution of events ≤ t,-3,-4.528721
5,Sum of contribution of events ≤ t,-2,-4.553879
3,Sum of contribution of events ≤ t,-1,-4.571843
96,Sum of contribution of events > t,-48,-4.590675


In [46]:
number_of_events_negative = negative_instance_data.shape[1]
pruning_negative_plot = plot_temp_coalition_pruning(coal_plot_negative_data, coal_prun_negative_idx, plot_limit=number_of_events_negative)
pruning_negative_plot

the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.


alt.LayerChart(...)

## Event level explanations

In [ ]:
# The article https://medium.com/feedzaitech/timeshap-explaining-recurrent-models-through-sequence-perturbations-41f2324bfe5f#27d4
# nsamples:  The number of coalitions for TimeSHAP to sample.
#   TimeSHAP needs to consider different combinations of these 35 features being present or absent
#   In theory, there are 2^35 possible combinations, but this would be computationally infeasible.
#   The number of samples can be adjusted to increase the granularity of the explanation.
#   A common starting point is around 2048-32000 samples.
# rs: Random seed for reproducibility.

event_dict = {'rs': 42, 'nsamples': 500000}
event_data = local_event(f_hs, positive_instance_data, event_dict, positive_instance_id, sequence_id_feat, average_event, pruning_negative_idx)

In [101]:
event_data.sort_values(by='Shapley Value', ascending=False)

,Random seed,NSamples,Feature,Shapley Value
46,42,500000,Event -47,3.133147
47,42,500000,Event -48,3.040634
45,42,500000,Event -46,1.814049
30,42,500000,Event -31,1.762642
44,42,500000,Event -45,1.293733
28,42,500000,Event -29,1.041201
43,42,500000,Event -44,0.991386
29,42,500000,Event -30,0.951117
31,42,500000,Event -32,0.818098
32,42,500000,Event -33,0.776479


In [102]:
event_plot = plot_event_heatmap(event_data)
# event_plot.properties(
#     width=1000,
#     height=900
#     )
event_plot

the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.


alt.LayerChart(...)

In [48]:
event_dict = {'rs': 42, 'nsamples': 500000}
event_data_negative = local_event(f_hs, negative_instance_data, event_dict, negative_instance_id, sequence_id_feat, average_event, pruning_negative_idx)

event_plot_negative = plot_event_heatmap(event_data_negative)
# event_plot_negative.properties(
#     width=1000,
#     height=900
#     )
event_plot_negative

the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.


alt.LayerChart(...)

## Feature level explanations

In [103]:
# 1. First, let's make a copy of your data to avoid modifying the original
normalized_data = positive_instance_data.copy()

# 2. Normalize the Hours column (index 0) to be between 0 and 1
normalized_data[:, :, 0] = positive_instance_data[:, :, 0] / 47.0  # Since Hours goes from 0 to 47

# 3. Now try TimeSHAP with the normalized data
feature_dict = {
    'rs': 42, 
    'nsamples': 500000,
    'feature_names': model_features,  # Keep all original feature names
    'plot_features': plot_features   # Use all features for plotting
}

feature_data = local_feat(f_hs, normalized_data, feature_dict, positive_instance_id, sequence_id_feat, average_event, pruning_idx)

In [104]:
feature_data.sort_values(by='Shapley Value', ascending=False)

,Random seed,NSamples,Feature,Shapley Value
18,42,500000,uo_rt_6hr,17.292213
24,42,500000,F,13.374061
22,42,500000,vent,0.309348
16,42,500000,uo_rt_12hr,0.279751
11,42,500000,potassium_avg,0.255685
5,42,500000,creat,0.237071
9,42,500000,hematocrit_avg,0.215792
19,42,500000,wbc_avg,0.118721
12,42,500000,resprate_mean,0.107348
1,42,500000,aniongap_avg,0.072423


In [105]:
feature_plot = plot_feat_barplot(feature_data, feature_dict.get('top_feats'), feature_dict.get('plot_features'))
feature_plot

the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.


alt.LayerChart(...)

In [49]:
# 1. First, let's make a copy of your data to avoid modifying the original
normalized_negative_data = negative_instance_data.copy()

# 2. Normalize the Hours column (index 0) to be between 0 and 1
normalized_negative_data[:, :, 0] = negative_instance_data[:, :, 0] / 47.0  # Since Hours goes from 0 to 47

# 3. Now try TimeSHAP with the normalized data
feature_dict = {
    'rs': 42, 
    'nsamples': 500000,
    'feature_names': model_features,  # Keep all original feature names
    'plot_features': plot_features   # Use all features for plotting
}

feature_negative_data = local_feat(f_hs, normalized_negative_data, feature_dict, negative_instance_id, sequence_id_feat, average_event, pruning_negative_idx)

feature_negative_plot = plot_feat_barplot(feature_negative_data, feature_dict.get('top_feats'), feature_dict.get('plot_features'))
feature_negative_plot

the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.


alt.LayerChart(...)

## Cell event explanations

In [36]:
cell_dict = {'rs': 42, 'nsamples': 100000, 'top_x_events': 5, 'top_x_feats': 5}
cell_data = local_cell_level(f_hs, positive_instance_data, cell_dict, event_data, feature_data, positive_instance_id, sequence_id_feat, average_event, pruning_idx)
feat_names = list(feature_data['Feature'].values)[:-1] # exclude pruned events
cell_plot = plot_cell_level(cell_data, feat_names, feature_dict.get('plot_features'))
cell_plot

the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.


alt.LayerChart(...)

# Global explanations

In [25]:
from timeshap.explainer import prune_all, pruning_statistics, event_explain_all, feat_explain_all
from timeshap.plot import plot_global_event, plot_global_feat

In [21]:
# Explain all

from timeshap.explainer import global_report 

pos_dataset = df_testdata[df_testdata['label'] == 1]
schema = schema = list(pos_dataset.columns)
pruning_dict = {'tol': [0.05, 0.075], 'path': 'outputs/prun_all.csv'}
event_dict = {'path': 'outputs/event_all.csv', 'rs': 42, 'nsamples': 32000}
feature_dict = {'path': 'outputs/feature_all.csv', 'rs': 42, 'nsamples': 32000, 'feature_names': model_features, 'plot_features': plot_features}
prun_stats, global_plot = global_report(f_hs, pos_dataset, pruning_dict, event_dict, feature_dict, average_event, model_features, schema, sequence_id_feat, time_feat)
prun_stats

The defined path for pruning data already exists and the append option is turned off. TimeSHAP will only read from this file and will not create new explanation data
The defined path for event explanations already exists and the append option is turned off. TimeSHAP will only read from this file and will not create new explanation data
The defined path for feature explanations already exists and the append option is turned off. TimeSHAP will only read from this file and will not create new explanation data
Calculating pruning algorithm
Calculating event data
Calculating feat data
Calculating pruning indexes


The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


,Tolerance,Mean,Std
0,0.05,45.078903,8.495400
1,0.075,44.447197,9.230182
2,No Pruning,48.000000,0.000000


In [22]:
global_plot

MaxRowsError: The number of rows in your dataset is greater than the maximum allowed (5000). For information on how to plot larger datasets in Altair, see the documentation

alt.VConcatChart(...)

## Pruning statistics

In [26]:
pos_dataset = df_testdata[df_testdata['label'] == 1]
schema = schema = list(pos_dataset.columns)
sequence_id_feat = 'instance'
time_feat = 'Hours'

In [27]:
average_event

,Hours,aniongap_avg,bicarbonate_avg,bun_avg,chloride_avg,creat,diasbp_mean,glucose_avg,heartrate_mean,hematocrit_avg,...,M,white_group,ELECTIVE,URGENT,black_african_group,asian_group,hispanic_group,unknown_group,native_group,other_group
0,23.999861,0.368421,0.5,0.147059,0.558824,0.083333,0.394366,0.198212,0.414634,0.444043,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
pruning_dict = {'tol': [0.05, 0.075], 'path': 'outputs/prun_all.csv'}
prun_indexes = prune_all(f_hs, pos_dataset, pruning_dict, average_event, model_features, schema, sequence_id_feat, time_feat)
pruning_stats = pruning_statistics(prun_indexes, pruning_dict.get('tol'))
pruning_stats

,Tolerance,Mean,Std
0,0.05,45.078903,8.495400
1,0.075,44.447197,9.230182
2,No Pruning,48.000000,0.000000


## Global event level

In [28]:
print("Sequence length:", pos_dataset.shape)  # Assuming first dimension is sequence length

Sequence length: (199536, 37)


In [51]:

# pruning_dict = {'tol': [0.05, 0.075], 'path': 'outputs/prun_all.csv'}
# feature_dict = {'path': 'outputs/feature_all.csv', 'rs': 42, 'nsamples': 32000, 'feature_names': model_features, 'plot_features': plot_features}

In [52]:
event_dict = {'path': 'outputs/event_all.csv', 'rs': 42, 'nsamples': 32000}
event_data = event_explain_all(f_hs, pos_dataset, event_dict, prun_indexes, average_event, model_features, schema, sequence_id_feat, time_feat)
event_data.sort_values(by='Shapley Value', ascending=False)

,Random Seed,NSamples,Event,Shapley Value,t (event index),Entity,Tolerance
9523,42,32000,Event -37,12.019177,-36,142.0,0.075
9480,42,32000,Event -37,12.019177,-36,142.0,0.050
15948,42,32000,Event -34,10.986948,-33,236.0,0.075
15911,42,32000,Event -34,10.986948,-33,236.0,0.050
3904,42,32000,Event -41,10.787915,-40,59.0,0.075
...,...,...,...,...,...,...,...
2028,42,32000,Event -19,-0.740854,-18,27.0,0.075
13407,42,32000,Pruned Events,-0.987069,1,205.0,0.050
13414,42,32000,Pruned Events,-0.987069,1,205.0,0.075
3902,42,32000,Event -39,-1.000778,-38,59.0,0.075


In [53]:
event_data.iloc[45:50]
# event_data.head()

,Random Seed,NSamples,Event,Shapley Value,t (event index),Entity,Tolerance
45,42,32000,Event -46,-0.072633,-45,1.0,0.050
46,42,32000,Event -47,-0.080009,-46,1.0,0.050
47,42,32000,Event -48,-0.066113,-47,1.0,0.050
48,42,32000,Event -1,-0.013712,0,1.0,0.075
49,42,32000,Event -2,0.019263,-1,1.0,0.075


In [56]:
plot_parameters = {
    'height': 250,
    'width': 460,
    'axis_lims': [-1.1, 12],
    't_limit': -50
}
event_global_plot = plot_global_event(event_data, plot_parameters=plot_parameters)
event_global_plot

MaxRowsError: The number of rows in your dataset is greater than the maximum allowed (5000). For information on how to plot larger datasets in Altair, see the documentation

alt.VConcatChart(...)

## Global feature-level

In [59]:
feat_data.sort_values(by='Shapley Value', ascending=False)

,Random Seed,NSamples,Feature,Shapley Value,Entity,Tolerance
5237,42,32000,uo_rt_6hr,28.569485,101.0,0.050
5272,42,32000,uo_rt_6hr,28.569485,101.0,0.075
11932,42,32000,uo_rt_6hr,27.802951,231.0,0.050
11967,42,32000,uo_rt_6hr,27.802951,231.0,0.075
12109,42,32000,uo_rt_6hr,27.585601,233.0,0.075
...,...,...,...,...,...,...
453,42,32000,white_group,-9.226982,8.0,0.050
488,42,32000,white_group,-9.226982,8.0,0.075
7151,42,32000,M,-10.415589,139.0,0.050
1298,42,32000,M,-10.514513,24.0,0.050


In [31]:
feature_dict = {'path': 'outputs/feature_all.csv', 'rs': 42, 'nsamples': 32000, 'feature_names': model_features, 'plot_features': plot_features, }
feat_data = feat_explain_all(f_hs, pos_dataset, feature_dict, prun_indexes, average_event, model_features, schema, sequence_id_feat, time_feat)
feat_global_plot = plot_global_feat(feat_data, **feature_dict)
feat_global_plot

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.


alt.VConcatChart(...)